In [ ]:
%%writefile requirements.txt
gradio>=4.20.0
huggingface_hub
numpy
pillow
tensorflow-cpu

Writing requirements.txt


In [ ]:
import time
import os
import numpy as np
from PIL import Image
import tensorflow as tf
import gradio as gr
from huggingface_hub import hf_hub_download

# 1. Model Repository Information
MODEL_REPO = "iamabhayaditya/InceptionResNetV2-UNet-MBRSC"

MODEL_CONFIGS = {
    "TFLite INT8 (Dynamic Quantized - 35.89 MB, Edge-Optimized)": {
        "filename": "InceptionResNetV2-UNet_int8.tflite",
        "size": "35.89 MB",
        "precision": "INT8 Dynamic PTQ"
    },
    "TFLite FP32 (Full Precision - 140.36 MB)": {
        "filename": "InceptionResNetV2-UNet_fp32.tflite",
        "size": "140.36 MB",
        "precision": "FP32 Float"
    }
}

# 2. Color Mapping (Exact MBRSC Class Dictionary)
COLOR_MAP = np.array([
    [60, 16, 152],   # 0: Building
    [132, 41, 246],  # 1: Land
    [110, 193, 228], # 2: Road
    [254, 221, 58],  # 3: Vegetation
    [226, 169, 41],  # 4: Water
    [155, 155, 155], # 5: Unlabeled
], dtype=np.uint8)

loaded_interpreters = {}

def get_interpreter(model_key):
    if model_key not in loaded_interpreters:
        filename = MODEL_CONFIGS[model_key]["filename"]
        # Check local folder first; if not present, download from HF Hub
        if os.path.exists(filename):
            model_path = filename
        else:
            print(f"Downloading {filename} from {MODEL_REPO}...")
            model_path = hf_hub_download(repo_id=MODEL_REPO, filename=filename)

        interpreter = tf.lite.Interpreter(model_path=model_path, num_threads=4)
        interpreter.allocate_tensors()
        loaded_interpreters[model_key] = interpreter
    return loaded_interpreters[model_key]

def segment_satellite_image(input_img, model_choice, opacity):
    if input_img is None:
        return None, None, "⚠️ Please upload or click an example satellite image."

    # Preprocess
    orig_img = Image.fromarray(input_img).convert("RGB")
    resized_img = orig_img.resize((512, 512))
    img_np = np.array(resized_img, dtype=np.float32) / 255.0
    input_tensor = np.expand_dims(img_np, axis=0)

    # Inference & Timing
    interpreter = get_interpreter(model_choice)
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    start_time = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], input_tensor)
    interpreter.invoke()
    predictions = interpreter.get_tensor(output_details[0]['index'])
    latency_ms = (time.perf_counter() - start_time) * 1000

    # Postprocess
    pred_mask = np.argmax(predictions[0], axis=-1)  # (512, 512)
    mask_rgb = COLOR_MAP[pred_mask]                 # (512, 512, 3)

    # Generate Blended Overlay
    orig_np = np.array(resized_img, dtype=np.uint8)
    blended = (orig_np * (1.0 - opacity) + mask_rgb * opacity).astype(np.uint8)

    config = MODEL_CONFIGS[model_choice]
    badge = f"""
    ### ⚡ Live Inference Benchmark
    | Metric | Measurement |
    | :--- | :--- |
    | **Execution Latency** | **`{latency_ms:.1f} ms`** (CPU Multi-threaded) |
    | **Active Precision** | `{config['precision']}` |
    | **Model Footprint** | `{config['size']}` |
    """

    return blended, mask_rgb, badge

# 3. UI Layout
legend_html = """
<div style="display: flex; flex-wrap: wrap; gap: 8px; margin: 12px 0;">
    <span style="background-color: rgb(226, 169, 41); color: black; padding: 4px 10px; border-radius: 4px; font-weight: bold;">🌊 Water</span>
    <span style="background-color: rgb(254, 221, 58); color: black; padding: 4px 10px; border-radius: 4px; font-weight: bold;">🌿 Vegetation</span>
    <span style="background-color: rgb(132, 41, 246); color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold;">🏜️ Land</span>
    <span style="background-color: rgb(110, 193, 228); color: black; padding: 4px 10px; border-radius: 4px; font-weight: bold;">🛣️ Road</span>
    <span style="background-color: rgb(60, 16, 152); color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold;">🏢 Building</span>
    <span style="background-color: rgb(155, 155, 155); color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold;">❓ Unlabeled</span>
</div>
"""

with gr.Blocks(title="Dubai Satellite Segmentation & Edge Optimization") as demo:
    gr.Markdown("# 🛰️ Dubai Satellite Imagery Semantic Segmentation")
    gr.Markdown("Real-time aerial land-cover segmentation powered by **InceptionResNetV2-UNet** with post-training edge quantization.")
    gr.HTML(legend_html)

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="numpy", label="Upload Satellite Patch (512x512)")
            model_dropdown = gr.Dropdown(
                choices=list(MODEL_CONFIGS.keys()),
                value=list(MODEL_CONFIGS.keys())[0],
                label="Select Deployment Precision"
            )
            opacity_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.6, step=0.05, label="Mask Overlay Opacity")
            submit_btn = gr.Button("🚀 Segment Terrain", variant="primary")

            # Clickable Example Thumbnails (Only loads images without overriding model dropdown)
            example_files = [
                os.path.join("examples", f) for f in os.listdir("examples")
                if f.endswith((".png", ".jpg", ".jpeg"))
            ] if os.path.exists("examples") else []

            if example_files:
                gr.Examples(
                    examples=[[img] for img in example_files],
                    inputs=[input_image],
                    label="Click a Sample Satellite Patch to Test:"
                )

        with gr.Column(scale=1):
            with gr.Tab("Blended Overlay"):
                output_blended = gr.Image(type="numpy", label="Satellite Image + Segmentation Mask")
            with gr.Tab("Raw Mask"):
                output_mask = gr.Image(type="numpy", label="Class Prediction Mask")
            output_badge = gr.Markdown()

    submit_btn.click(
        fn=segment_satellite_image,
        inputs=[input_image, model_dropdown, opacity_slider],
        outputs=[output_blended, output_mask, output_badge]
    )

if __name__ == "__main__":
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://77e6c498f7400acc58.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
%%writefile app.py
import time
import os
import numpy as np
from PIL import Image
import tensorflow as tf
import gradio as gr
from huggingface_hub import hf_hub_download

# 1. Model Repository Information
MODEL_REPO = "iamabhayaditya/InceptionResNetV2-UNet-MBRSC"

MODEL_CONFIGS = {
    "TFLite INT8 (Dynamic Quantized - 35.89 MB, Edge-Optimized)": {
        "filename": "InceptionResNetV2-UNet_int8.tflite",
        "size": "35.89 MB",
        "precision": "INT8 Dynamic PTQ"
    },
    "TFLite FP32 (Full Precision - 140.36 MB)": {
        "filename": "InceptionResNetV2-UNet_fp32.tflite",
        "size": "140.36 MB",
        "precision": "FP32 Float"
    }
}

# 2. Color Mapping (Exact MBRSC Class Dictionary)
COLOR_MAP = np.array([
    [60, 16, 152],   # 0: Building
    [132, 41, 246],  # 1: Land
    [110, 193, 228], # 2: Road
    [254, 221, 58],  # 3: Vegetation
    [226, 169, 41],  # 4: Water
    [155, 155, 155], # 5: Unlabeled
], dtype=np.uint8)

loaded_interpreters = {}

def get_interpreter(model_key):
    if model_key not in loaded_interpreters:
        filename = MODEL_CONFIGS[model_key]["filename"]
        if os.path.exists(filename):
            model_path = filename
        else:
            print(f"Downloading {filename} from {MODEL_REPO}...")
            model_path = hf_hub_download(repo_id=MODEL_REPO, filename=filename)

        interpreter = tf.lite.Interpreter(model_path=model_path, num_threads=4)
        interpreter.allocate_tensors()
        loaded_interpreters[model_key] = interpreter
    return loaded_interpreters[model_key]

def segment_satellite_image(input_img, model_choice, opacity):
    if input_img is None:
        return None, None, "⚠️ Please upload or click an example satellite image."

    # Preprocessing
    orig_img = Image.fromarray(input_img).convert("RGB")
    resized_img = orig_img.resize((512, 512))
    img_np = np.array(resized_img, dtype=np.float32) / 255.0
    input_tensor = np.expand_dims(img_np, axis=0)

    # Inference & Timing
    interpreter = get_interpreter(model_choice)
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    start_time = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], input_tensor)
    interpreter.invoke()
    predictions = interpreter.get_tensor(output_details[0]['index'])
    latency_ms = (time.perf_counter() - start_time) * 1000

    # Postprocessing
    pred_mask = np.argmax(predictions[0], axis=-1)  # (512, 512)
    mask_rgb = COLOR_MAP[pred_mask]                 # (512, 512, 3)

    # Blended Overlay
    orig_np = np.array(resized_img, dtype=np.uint8)
    blended = (orig_np * (1.0 - opacity) + mask_rgb * opacity).astype(np.uint8)

    config = MODEL_CONFIGS[model_choice]
    badge = f"""
    ### ⚡ Live Inference Benchmark
    | Metric | Measurement |
    | :--- | :--- |
    | **Execution Latency** | **`{latency_ms:.1f} ms`** (CPU Multi-threaded) |
    | **Active Precision** | `{config['precision']}` |
    | **Model Footprint** | `{config['size']}` |
    """

    return blended, mask_rgb, badge

legend_html = """
<div style="display: flex; flex-wrap: wrap; gap: 8px; margin: 12px 0;">
    <span style="background-color: rgb(226, 169, 41); color: black; padding: 4px 10px; border-radius: 4px; font-weight: bold;">🌊 Water</span>
    <span style="background-color: rgb(254, 221, 58); color: black; padding: 4px 10px; border-radius: 4px; font-weight: bold;">🌿 Vegetation</span>
    <span style="background-color: rgb(132, 41, 246); color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold;">🏜️ Land</span>
    <span style="background-color: rgb(110, 193, 228); color: black; padding: 4px 10px; border-radius: 4px; font-weight: bold;">🛣️ Road</span>
    <span style="background-color: rgb(60, 16, 152); color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold;">🏢 Building</span>
    <span style="background-color: rgb(155, 155, 155); color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold;">❓ Unlabeled</span>
</div>
"""

with gr.Blocks(title="Dubai Satellite Segmentation & Edge Optimization") as demo:
    gr.Markdown("# 🛰️ Dubai Satellite Imagery Semantic Segmentation")
    gr.Markdown("Real-time aerial land-cover segmentation powered by **InceptionResNetV2-UNet** with post-training edge quantization.")
    gr.HTML(legend_html)

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="numpy", label="Upload Satellite Patch (512x512)")
            model_dropdown = gr.Dropdown(
                choices=list(MODEL_CONFIGS.keys()),
                value=list(MODEL_CONFIGS.keys())[0],
                label="Select Deployment Precision"
            )
            opacity_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.6, step=0.05, label="Mask Overlay Opacity")
            submit_btn = gr.Button("🚀 Segment Terrain", variant="primary")

            # Clickable Example Thumbnails
            example_files = [
                os.path.join("examples", f) for f in os.listdir("examples")
                if f.endswith((".png", ".jpg", ".jpeg"))
            ] if os.path.exists("examples") else []

            if example_files:
                gr.Examples(
                    examples=[[img] for img in example_files],
                    inputs=[input_image],
                    label="Click a Sample Satellite Patch to Test:"
                )

        with gr.Column(scale=1):
            with gr.Tab("Blended Overlay"):
                output_blended = gr.Image(type="numpy", label="Satellite Image + Segmentation Mask")
            with gr.Tab("Raw Mask"):
                output_mask = gr.Image(type="numpy", label="Class Prediction Mask")
            output_badge = gr.Markdown()

    submit_btn.click(
        fn=segment_satellite_image,
        inputs=[input_image, model_dropdown, opacity_slider],
        outputs=[output_blended, output_mask, output_badge]
    )

if __name__ == "__main__":
    demo.launch()

Writing app.py


In [ ]:
import os
from huggingface_hub import HfApi
from google.colab import userdata

# Retrieve Hugging Face token safely from Colab Secrets
try:
    HF_TOKEN = userdata.get('hf-upload-token')
except Exception as e:
    raise ValueError(
        "❌ Error retrieving 'hf-upload-token' from Colab Secrets.\n"
        "Please ensure you have added a secret key named 'hf-upload-token' \n"
        "in the Secrets panel (🔑 icon on the left sidebar) and granted notebook access."
    ) from e

SPACE_REPO = "iamabhayaditya/InceptionResNetV2-UNet-MBRSC-demo"

# Initialize HfApi with the retrieved token
api = HfApi(token=HF_TOKEN)

print(f"🚀 Deploying to Space: https://huggingface.co/spaces/{SPACE_REPO} ...\n")

# 1. Upload requirements.txt
api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=SPACE_REPO,
    repo_type="space"
)
print("✅ Uploaded requirements.txt")

# 2. Upload app.py
api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id=SPACE_REPO,
    repo_type="space"
)
print("✅ Uploaded app.py")

# 3. Upload the examples folder if it exists locally in Colab
if os.path.exists("examples"):
    api.upload_folder(
        folder_path="examples",
        path_in_repo="examples",
        repo_id=SPACE_REPO,
        repo_type="space"
    )
    print("✅ Uploaded examples folder with sample satellite images!")

print("\n🎉 Deployment trigger sent!")
print(f"👉 Open your live Space here: https://huggingface.co/spaces/{SPACE_REPO}")

🚀 Deploying to Space: https://huggingface.co/spaces/iamabhayaditya/InceptionResNetV2-UNet-MBRSC-demo ...

✅ Uploaded requirements.txt
✅ Uploaded app.py
✅ Uploaded examples folder with sample satellite images!

🎉 Deployment trigger sent!
👉 Open your live Space here: https://huggingface.co/spaces/iamabhayaditya/InceptionResNetV2-UNet-MBRSC-demo
